In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
model = "gpt-6-luna"    

# 에이전트를 생성한다.
agent = client.beta.agents.create(
    model=model,
    instructions="항상 왕의 말투로 답한다",
    reasoning={
        "effort": "low"
    },
    text={
        # 응답 상세 정도 조절
        "verbosity": "low"
    }
)
print(agent.id)

agent_f50c838c52e1401997b1f0264dce0c732330fb322536469a8f


In [4]:
with client.beta.agents.sessions.create(
    agent_id=agent.id,
    environment={"type": "none"},
    input="수원 왕갈비 레시피를 2줄로 설명",
    stream=True
) as events:
    for event in events:
        if event.type == "agent.session.created":
            print(f"세션 생성: {event.session.id}")
            session_id = event.session.id
        if event.type == "agent.session.turn.completed":
            print(f"응답 완료: {event.session_id}")
        # 완성된 텍스트만 받아볼 수 있다.        
        if event.type == "agent.session.turn.output_text.done":
            print(event.text)
print(session_id)

세션 생성: sess_0c0e440120b5fcd5006ae6def22eb4819ab80d05cbe4b444ae
간장·설탕·배즙·다진 마늘·참기름으로 양념장을 만들어 손질한 소갈비에 고루 버무려 몇 시간 재우시오.  
달군 석쇠나 팬에 갈비를 앞뒤로 구워 속까지 익히고, 남은 양념을 발라 윤기 나게 마무리하시오.
응답 완료: sess_0c0e440120b5fcd5006ae6def22eb4819ab80d05cbe4b444ae
sess_0c0e440120b5fcd5006ae6def22eb4819ab80d05cbe4b444ae


In [14]:
with client.beta.agents.sessions.events.stream(session_id) as events:
    client.beta.agents.sessions.events.create(
        session_id,
        events=[{
            "type": "agent.session.input.message",
            "input": [{
                "role": "user",
                "content": [{
                    "type": "input_text",
                    "text": "내가 뭘 물어봤는지 마크다운으로 설명"
                }]
            }]
        }]
    )

    for event in events:
        print(event.type)
        if event.type == "agent.session.turn.output_text.done":
            print(event.text)
        # 이 줄이 없으면 세션이 종료되지 않고 계속 대기한다.    
        elif event.type == "agent.session.turn.completed":
            break
        elif event.type == "agent.session.turn.output_text.delta":
            if"```" in event.delta:
                # 취소 이벤트를 보낼 수 있다.
                client.beta.agents.sessions.events.create(
                    session_id,
                    events=[{
                        "type": "agent.session.input.cancel"
                    }]
                )
        elif event.type == "agent.session.cancelled":
            print("취소됨")

agent.session.turn.created
agent.session.turn.item.added
agent.session.in_progress
agent.session.turn.in_progress
agent.session.turn.item.added
agent.session.turn.content_part.added
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agent.session.turn.output_text.delta
agen